In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_prep import prepare_sales_data, prepare_ppc_data




pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
DB_PATH = "/Users/ricardolugo/Library/CloudStorage/OneDrive-Personal/Hasten/Reports/SQLite/litet.db"

In [3]:
def load_table(table_name: str) -> pd.DataFrame:
    conn = sqlite3.connect(DB_PATH)

    df = pd.read_sql(
        f'SELECT * FROM "{table_name}"',
        conn
    )

    conn.close()

    return df


def load_orders():
    return load_table("orders")


def load_asins():
    return load_table("asins")


def load_ppc():
    return load_table("ppc")

In [4]:
orders_raw = load_orders()
asins_raw = load_asins()
ppc_raw = load_ppc()

In [5]:
print("ORDERS:", orders_raw.shape)
print("ASINS:", asins_raw.shape)
print("PPC:", ppc_raw.shape)

ORDERS: (10175, 37)
ASINS: (12, 6)
PPC: (23428, 29)


In [6]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

conn.close()

tables

,name
0,orders_backup_duplicated
1,fba_inventory
2,asins
3,ppc
4,transactions
5,inventory_snapshots
6,orders
7,orders_clean


In [7]:
sales_df = prepare_sales_data(
    orders_raw,
    asins_raw
)

ppc_df = prepare_ppc_data(
    ppc_raw,
    campaign_filter="LITET"
)

/Users/ricardolugo/Documents/dev/hasten/litet_sales_bot/data_prep.py:160: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ppc["date"] = pd.to_datetime(


In [8]:
weekly_sales = (
    sales_df
    .groupby(
        pd.Grouper(
            key="purchase_date",
            freq="W-MON"
        )
    )
    .agg(
        revenue=("revenue", "sum"),
        orders=("amazon-order-id", "nunique"),
        units=("units_sold_amazon", "sum"),
        pairs=("pairs_sold", "sum")
    )
    .reset_index()
)

weekly_sales.tail(20)

,purchase_date,revenue,orders,units,pairs
50,2026-02-09 00:00:00+00:00,829.67,28,33,65
51,2026-02-16 00:00:00+00:00,1117.53,36,47,87
52,2026-02-23 00:00:00+00:00,713.72,27,28,56
53,2026-03-02 00:00:00+00:00,884.68,26,32,70
54,2026-03-09 00:00:00+00:00,1260.51,46,49,99
55,2026-03-16 00:00:00+00:00,1530.35,55,65,119
56,2026-03-23 00:00:00+00:00,1294.42,45,58,100
57,2026-03-30 00:00:00+00:00,1071.60,39,43,88
58,2026-04-06 00:00:00+00:00,1405.58,41,42,105
59,2026-04-13 00:00:00+00:00,1474.50,43,50,108


In [9]:
weekly_sales["revenue_4wk_avg"] = (
    weekly_sales["revenue"]
    .rolling(4)
    .mean()
)

weekly_sales[
    ["purchase_date", "revenue", "revenue_4wk_avg"]
].tail(15)

,purchase_date,revenue,revenue_4wk_avg
55,2026-03-16 00:00:00+00:00,1530.35,1097.3150
56,2026-03-23 00:00:00+00:00,1294.42,1242.4900
57,2026-03-30 00:00:00+00:00,1071.60,1289.2200
58,2026-04-06 00:00:00+00:00,1405.58,1325.4875
59,2026-04-13 00:00:00+00:00,1474.50,1311.5250
60,2026-04-20 00:00:00+00:00,1400.50,1338.0450
61,2026-04-27 00:00:00+00:00,1450.50,1432.7700
62,2026-05-04 00:00:00+00:00,1571.47,1474.2425
63,2026-05-11 00:00:00+00:00,2093.39,1628.9650
64,2026-05-18 00:00:00+00:00,1641.39,1689.1875


In [10]:
monthly_sales = (
    sales_df
    .assign(
        month=sales_df["purchase_date"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        revenue=("revenue","sum"),
        orders=("amazon-order-id","nunique"),
        pairs=("pairs_sold","sum")
    )
)

monthly_sales.tail(12)

/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_50746/4088553965.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  month=sales_df["purchase_date"].dt.to_period("M")


,revenue,orders,pairs
month,,,
2025-07,2424.11,140,198
2025-08,2263.94,95,182
2025-09,1735.09,83,136
2025-10,3698.43,143,305
2025-11,3742.90,136,349
2025-12,2827.85,100,229
2026-01,2296.10,78,178
2026-02,3383.65,114,266
2026-03,5534.73,197,435


In [11]:
weekly_sales.tail(10)

,purchase_date,revenue,orders,units,pairs,revenue_4wk_avg
60,2026-04-20 00:00:00+00:00,1400.50,45,50,103,1338.0450
61,2026-04-27 00:00:00+00:00,1450.50,43,51,113,1432.7700
62,2026-05-04 00:00:00+00:00,1571.47,48,53,117,1474.2425
63,2026-05-11 00:00:00+00:00,2093.39,58,61,159,1628.9650
64,2026-05-18 00:00:00+00:00,1641.39,53,62,124,1689.1875
65,2026-05-25 00:00:00+00:00,1690.39,55,61,124,1749.1600
66,2026-06-01 00:00:00+00:00,1336.62,38,38,101,1690.4475
67,2026-06-08 00:00:00+00:00,989.59,38,42,72,1414.4975
68,2026-06-15 00:00:00+00:00,1399.56,44,45,104,1354.0400
69,2026-06-22 00:00:00+00:00,459.82,16,18,33,1046.3975


In [12]:
ppc_df["date"] = pd.to_datetime(ppc_df["date"])

weekly_ppc = (
    ppc_df
    .groupby(
        pd.Grouper(
            key="date",
            freq="W-MON"
        )
    )
    .agg(
        impressions=("impressions","sum"),
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
    .reset_index()
)

In [13]:
weekly_ppc["cpc"] = (
    weekly_ppc["spend"] /
    weekly_ppc["clicks"]
)

weekly_ppc["cvr"] = (
    weekly_ppc["orders"] /
    weekly_ppc["clicks"]
)

weekly_ppc["acos"] = (
    weekly_ppc["spend"] /
    weekly_ppc["sales"]
)

weekly_ppc["roas"] = (
    weekly_ppc["sales"] /
    weekly_ppc["spend"]
)

In [14]:
weekly_ppc.tail(15)

,date,impressions,clicks,spend,sales,orders,cpc,cvr,acos,roas
51,2026-03-16,14181,282,710.43,824.69,28,2.519255,0.099291,0.861451,1.160832
52,2026-03-23,10301,155,268.70,532.80,18,1.733548,0.116129,0.504317,1.982881
53,2026-03-30,10574,150,271.78,616.81,19,1.811867,0.126667,0.440622,2.269519
54,2026-04-06,15292,207,394.35,685.81,19,1.905072,0.091787,0.575013,1.739090
55,2026-04-13,9254,171,315.38,894.72,27,1.844327,0.157895,0.352490,2.836959
56,2026-04-20,13031,169,295.73,649.80,19,1.749882,0.112426,0.455109,2.197275
57,2026-04-27,12554,211,385.40,699.75,23,1.826540,0.109005,0.550768,1.815646
58,2026-05-04,11781,219,408.48,639.79,21,1.865205,0.095890,0.638459,1.566270
59,2026-05-11,10138,213,385.77,1417.64,36,1.811127,0.169014,0.272121,3.674832
60,2026-05-18,14140,222,378.11,785.76,23,1.703198,0.103604,0.481203,2.078125


In [15]:
good_period = ppc_df[
    (ppc_df["date"] >= "2026-05-05") &
    (ppc_df["date"] <= "2026-05-11")
]

bad_period = ppc_df[
    (ppc_df["date"] >= "2026-05-26") &
    (ppc_df["date"] <= "2026-06-01")
]

In [16]:
good_terms = (
    good_period
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        sales=("sales","sum"),
        orders=("orders","sum"),
        spend=("spend","sum")
    )
)

bad_terms = (
    bad_period
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        sales=("sales","sum"),
        orders=("orders","sum"),
        spend=("spend","sum")
    )
)

In [17]:
sales_df.groupby("type").agg(
    revenue=("revenue","sum"),
    pairs=("pairs_sold","sum")
)

,revenue,pairs
type,,
3-pack,29927.38,2442
6-pack,1229.82,114
single,17415.26,1267


In [18]:
sales_df.groupby(
    [
        pd.Grouper(key="purchase_date", freq="W-MON"),
        "type"
    ]
).agg(
    revenue=("revenue","sum")
)

revenue
purchase_date             type           
2025-02-24 00:00:00+00:00 single    69.95
2025-03-03 00:00:00+00:00 single    55.96
2025-03-10 00:00:00+00:00 single   265.81
2025-03-17 00:00:00+00:00 single   237.83
2025-03-24 00:00:00+00:00 single    41.97
...                                   ...
2026-06-15 00:00:00+00:00 6-pack    79.99
                          single   239.84
2026-06-22 00:00:00+00:00 3-pack   199.95
                          6-pack    79.99
                          single   179.88

[135 rows x 1 columns]

In [19]:
weekly_sales["week"] = (
    pd.to_datetime(weekly_sales["purchase_date"])
    .dt.tz_localize(None)
)

weekly_ppc["week"] = (
    pd.to_datetime(weekly_ppc["date"])
    .dt.tz_localize(None)
)

In [20]:
weekly = weekly_sales.merge(
    weekly_ppc,
    on="week",
    how="left",
    suffixes=("_total", "_ppc")
)

In [21]:
weekly["organic_sales"] = weekly["revenue"] - weekly["sales"]
weekly["ppc_share"] = weekly["sales"] / weekly["revenue"]

In [22]:
weekly[
    [
        "week",
        "revenue",
        "sales",
        "organic_sales",
        "ppc_share",
        "clicks",
        "spend",
        "orders_total",
        "orders_ppc",
        "cpc",
        "cvr",
        "acos",
        "roas"
    ]
].tail(15)

,week,revenue,sales,organic_sales,ppc_share,clicks,spend,orders_total,orders_ppc,cpc,cvr,acos,roas
55,2026-03-16,1530.35,824.69,705.66,0.538890,282.0,710.43,55,28.0,2.519255,0.099291,0.861451,1.160832
56,2026-03-23,1294.42,532.80,761.62,0.411613,155.0,268.70,45,18.0,1.733548,0.116129,0.504317,1.982881
57,2026-03-30,1071.60,616.81,454.79,0.575597,150.0,271.78,39,19.0,1.811867,0.126667,0.440622,2.269519
58,2026-04-06,1405.58,685.81,719.77,0.487920,207.0,394.35,41,19.0,1.905072,0.091787,0.575013,1.739090
59,2026-04-13,1474.50,894.72,579.78,0.606796,171.0,315.38,43,27.0,1.844327,0.157895,0.352490,2.836959
60,2026-04-20,1400.50,649.80,750.70,0.463977,169.0,295.73,45,19.0,1.749882,0.112426,0.455109,2.197275
61,2026-04-27,1450.50,699.75,750.75,0.482420,211.0,385.40,43,23.0,1.826540,0.109005,0.550768,1.815646
62,2026-05-04,1571.47,639.79,931.68,0.407128,219.0,408.48,48,21.0,1.865205,0.095890,0.638459,1.566270
63,2026-05-11,2093.39,1417.64,675.75,0.677198,213.0,385.77,58,36.0,1.811127,0.169014,0.272121,3.674832
64,2026-05-18,1641.39,785.76,855.63,0.478716,222.0,378.11,53,23.0,1.703198,0.103604,0.481203,2.078125


In [23]:
ppc_df.groupby("match_type").agg(
    clicks=("clicks","sum"),
    spend=("spend","sum"),
    sales=("sales","sum"),
    orders=("orders","sum")
).sort_values(
    "spend",
    ascending=False
)

,clicks,spend,sales,orders
match_type,,,,
BROAD,3573,6472.61,9811.26,341
PHRASE,1406,2675.98,4601.87,162
EXACT,643,1428.60,1879.36,64
-,591,853.59,1666.83,53
,10,26.16,67.98,2


In [24]:
ppc_df[
    [
        "keyword",
        "search_term",
        "match_type",
        "clicks",
        "sales",
        "orders"
    ]
].head(20)

,keyword,search_term,match_type,clicks,sales,orders
57,cycling socks,anti slip cycling sock,PHRASE,1,0.00,0
58,white cycling socks,white cycling socks,PHRASE,1,0.00,0
59,white cycling socks,white cycling socks,EXACT,1,0.00,0
60,road cycling socks,biking socks for men cycling,BROAD,1,0.00,0
61,mens cycling socks,cycling socks for men,BROAD,1,0.00,0
62,mens cycling socks,cycling socks for men,BROAD,1,0.00,0
63,mens cycling socks,cycling socks men,BROAD,1,0.00,0
64,mens cycling socks,mens cycling socks,BROAD,1,0.00,0
65,mens cycling socks,sock guy socks for men cycling,BROAD,1,0.00,0
66,mens cycling socks,mens cycling socks,PHRASE,1,0.00,0


In [25]:
good_period.groupby("match_type").agg(
    clicks=("clicks","sum"),
    spend=("spend","sum"),
    sales=("sales","sum"),
    orders=("orders","sum")
)

,clicks,spend,sales,orders
match_type,,,,
-,13,18.69,39.99,1
BROAD,146,252.52,749.80,20
EXACT,23,50.77,266.94,6
PHRASE,31,63.79,360.91,9


In [26]:
bad_period.groupby("match_type").agg(
    clicks=("clicks","sum"),
    spend=("spend","sum"),
    sales=("sales","sum"),
    orders=("orders","sum")
)

,clicks,spend,sales,orders
match_type,,,,
-,20,26.32,14.99,1
BROAD,108,175.20,134.96,4
EXACT,6,9.94,0.00,0
PHRASE,44,99.58,119.97,3


In [27]:
good_terms = (
    good_period
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
    .reset_index()
)

good_terms = good_terms.sort_values(
    "sales",
    ascending=False
)

In [28]:
bad_terms = (
    bad_period
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
    .reset_index()
)

bad_terms = bad_terms.sort_values(
    "sales",
    ascending=False
)

In [29]:
good_terms.head(20)

,search_term,clicks,spend,sales,orders
79,white cycling socks,24,50.79,226.95,5
30,cycling socks white,5,11.88,185.96,4
23,cycling socks,38,65.28,149.95,5
67,sock cycling,1,2.02,79.98,2
26,cycling socks for men,8,14.96,79.98,2
62,road bike socks for men,1,1.50,79.98,2
68,socks cycling,1,2.34,39.99,1
64,road cycling socks men,2,7.13,39.99,1
63,road cycling socks,1,2.02,39.99,1
21,cycling crew socks,1,1.50,39.99,1


In [30]:
bad_terms.head(20)

,search_term,clicks,spend,sales,orders
81,white cycling socks,9,14.59,79.98,2
29,cycling aero socks,3,4.36,39.99,1
52,litet cycling sock,11,32.80,39.99,1
6,aero socks cycling men,5,7.69,39.99,1
82,white cycling socks for men,1,2.40,39.99,1
77,thin summer cycling socks for men,1,1.95,14.99,1
9,b07zrjr667,3,4.69,14.99,1
0,aero cycling sock,1,1.15,0.00,0
65,pop cycling socks,1,1.70,0.00,0
64,performance cycling socks,1,1.75,0.00,0


In [31]:
term_comparison = (
    good_terms.merge(
        bad_terms,
        on="search_term",
        how="outer",
        suffixes=("_good", "_bad")
    )
    .fillna(0)
)

In [32]:
term_comparison["sales_delta"] = (
    term_comparison["sales_bad"]
    - term_comparison["sales_good"]
)

term_comparison["orders_delta"] = (
    term_comparison["orders_bad"]
    - term_comparison["orders_good"]
)

In [33]:
term_comparison.sort_values(
    "sales_delta"
).head(25)

,search_term,clicks_good,spend_good,sales_good,orders_good,clicks_bad,spend_bad,sales_bad,orders_bad,sales_delta,orders_delta
64,cycling socks white,5.0,11.88,185.96,4.0,13.0,27.49,0.00,0.0,-185.96,-4.0
50,cycling socks,38.0,65.28,149.95,5.0,17.0,25.92,0.00,0.0,-149.95,-5.0
138,white cycling socks,24.0,50.79,226.95,5.0,9.0,14.59,79.98,2.0,-146.97,-3.0
56,cycling socks for men,8.0,14.96,79.98,2.0,9.0,18.46,0.00,0.0,-79.98,-2.0
111,road bike socks for men,1.0,1.50,79.98,2.0,2.0,2.20,0.00,0.0,-79.98,-2.0
118,sock cycling,1.0,2.02,79.98,2.0,0.0,0.00,0.00,0.0,-79.98,-2.0
142,white cycling socks women,3.0,6.66,39.99,1.0,0.0,0.00,0.00,0.0,-39.99,-1.0
55,cycling socks fast drying,1.0,1.10,39.99,1.0,0.0,0.00,0.00,0.0,-39.99,-1.0
31,bicycle socks,3.0,4.50,39.99,1.0,0.0,0.00,0.00,0.0,-39.99,-1.0
135,white aero socks cycling,2.0,3.00,39.99,1.0,0.0,0.00,0.00,0.0,-39.99,-1.0


In [34]:
money_terms = [
    "cycling socks",
    "cycling socks white",
    "white cycling socks",
    "cycling socks for men"
]

ppc_df[
    ppc_df["search_term"].isin(money_terms)
].groupby(
    pd.Grouper(key="date", freq="W-MON")
).agg(
    clicks=("clicks","sum"),
    sales=("sales","sum"),
    orders=("orders","sum")
)

,clicks,sales,orders
date,,,
2025-03-24,5,13.99,1
2025-03-31,5,0.00,0
2025-04-07,3,0.00,0
2025-04-14,8,13.99,1
2025-04-21,5,13.99,1
...,...,...,...
2026-05-25,82,84.96,4
2026-06-01,48,79.98,2
2026-06-08,50,204.93,7


In [35]:
money_terms = [
    "cycling socks",
    "cycling socks white",
    "white cycling socks",
    "cycling socks for men"
]

In [36]:
money_df = ppc_df[
    ppc_df["search_term"].isin(money_terms)
].copy()

In [37]:
money_weekly = (
    money_df
    .groupby(
        [
            pd.Grouper(key="date", freq="W-MON"),
            "search_term"
        ]
    )
    .agg(
        clicks=("clicks","sum"),
        sales=("sales","sum"),
        orders=("orders","sum"),
        spend=("spend","sum")
    )
    .reset_index()
)

money_weekly

,date,search_term,clicks,sales,orders,spend
0,2025-03-24,cycling socks for men,2,0.00,0,5.41
1,2025-03-24,white cycling socks,3,13.99,1,6.53
2,2025-03-31,cycling socks,3,0.00,0,2.97
3,2025-03-31,cycling socks for men,1,0.00,0,2.61
4,2025-03-31,white cycling socks,1,0.00,0,1.65
...,...,...,...,...,...,...
223,2026-06-15,white cycling socks,25,79.98,2,50.75
224,2026-06-22,cycling socks,8,0.00,0,16.34
225,2026-06-22,cycling socks for men,9,0.00,0,26.37
226,2026-06-22,cycling socks white,1,39.99,1,1.89


In [38]:
money_weekly[
    money_weekly["date"] >= "2026-05-01"
].sort_values(
    ["date","sales"],
    ascending=[True,False]
)

,date,search_term,clicks,sales,orders,spend
198,2026-05-04,cycling socks white,8,79.98,2,16.92
197,2026-05-04,cycling socks for men,8,54.98,2,14.96
199,2026-05-04,white cycling socks,33,54.98,2,69.46
196,2026-05-04,cycling socks,43,44.97,3,80.52
203,2026-05-11,white cycling socks,24,226.95,5,50.79
202,2026-05-11,cycling socks white,5,185.96,4,11.88
200,2026-05-11,cycling socks,38,149.95,5,65.28
201,2026-05-11,cycling socks for men,8,79.98,2,14.96
205,2026-05-18,cycling socks for men,10,199.95,5,14.07
207,2026-05-18,white cycling socks,32,119.97,3,64.48


In [39]:
weekly_sku = (
    sales_df
    .groupby(
        [
            pd.Grouper(key="purchase_date", freq="W-MON"),
            "item_name"
        ]
    )
    .agg(
        revenue=("revenue","sum"),
        pairs=("pairs_sold","sum")
    )
    .reset_index()
)

In [40]:
weekly_sku[
    weekly_sku["purchase_date"] >= "2026-05-01"
]

,purchase_date,item_name,revenue,pairs
463,2026-05-04 00:00:00+00:00,"3-pack Black, Large/X-Large",79.98,6
464,2026-05-04 00:00:00+00:00,"3-pack White, Large/X-Large",319.92,24
465,2026-05-04 00:00:00+00:00,"3-pack White, Small/Medium",679.83,51
466,2026-05-04 00:00:00+00:00,"6-pack White, Large/X-Large",131.98,12
467,2026-05-04 00:00:00+00:00,"Black, Large/X-Large",14.99,1
...,...,...,...,...
539,2026-06-22 00:00:00+00:00,"Black, Large/X-Large",44.97,3
540,2026-06-22 00:00:00+00:00,"Black, Small/Medium",14.99,1
541,2026-06-22 00:00:00+00:00,"Blue, Large/X-Large",29.98,2
542,2026-06-22 00:00:00+00:00,"White, Large/X-Large",59.96,4


In [41]:
sku_weekly = (
    sales_df
    .groupby(
        [
            pd.Grouper(key="purchase_date", freq="W-MON"),
            "item_name"
        ]
    )
    .agg(
        revenue=("revenue","sum"),
        pairs=("pairs_sold","sum")
    )
    .reset_index()
)

pivot_revenue = sku_weekly.pivot_table(
    index="purchase_date",
    columns="item_name",
    values="revenue",
    aggfunc="sum",
    fill_value=0
)

pivot_revenue.tail(6)

item_name,"3-pack Black, Large/X-Large","3-pack Black, Small/Medium","3-pack White, Large/X-Large","3-pack White, Small/Medium","6-pack White, Large/X-Large","6-pack White, Small/Medium","Black, Large/X-Large","Black, Small/Medium","Blue, Large/X-Large","Blue, Small/Medium","White, Large/X-Large","White, Small/Medium"
purchase_date,,,,,,,,,,,,
2026-05-18 00:00:00+00:00,79.98,119.97,359.91,439.89,131.98,0.00,0.00,44.97,89.94,89.94,44.97,239.84
2026-05-25 00:00:00+00:00,79.98,39.99,399.90,639.84,65.99,0.00,44.97,29.98,89.94,74.95,134.91,89.94
2026-06-01 00:00:00+00:00,79.98,39.99,319.92,519.87,145.98,65.99,0.00,44.97,0.00,14.99,44.97,59.96
2026-06-08 00:00:00+00:00,79.98,79.98,159.96,279.93,0.00,0.00,29.98,44.97,29.98,29.98,134.91,119.92
2026-06-15 00:00:00+00:00,39.99,79.98,439.89,519.87,79.99,0.00,44.97,59.96,14.99,44.97,44.97,29.98
2026-06-22 00:00:00+00:00,0.00,0.00,0.00,199.95,79.99,0.00,44.97,14.99,29.98,0.00,59.96,29.98


In [42]:
top_skus = [
    "3-pack White, Small/Medium",
    "3-pack White, Large/X-Large",
    "White, Small/Medium",
    "White, Large/X-Large"
]

pivot_revenue[top_skus]

item_name,"3-pack White, Small/Medium","3-pack White, Large/X-Large","White, Small/Medium","White, Large/X-Large"
purchase_date,,,,
2025-02-24 00:00:00+00:00,0.00,0.00,0.00,13.99
2025-03-03 00:00:00+00:00,0.00,0.00,27.98,13.99
2025-03-10 00:00:00+00:00,0.00,0.00,27.98,41.97
2025-03-17 00:00:00+00:00,0.00,0.00,55.96,55.96
2025-03-24 00:00:00+00:00,0.00,0.00,0.00,13.99
...,...,...,...,...
2026-05-25 00:00:00+00:00,639.84,399.90,89.94,134.91
2026-06-01 00:00:00+00:00,519.87,319.92,59.96,44.97
2026-06-08 00:00:00+00:00,279.93,159.96,119.92,134.91


In [43]:
weekly["tacos"] = (
    weekly["spend"] /
    weekly["revenue"]
)

In [44]:
weekly[
    [
        "week",
        "revenue",
        "spend",
        "sales",
        "organic_sales",
        "tacos"
    ]
].tail(15)

,week,revenue,spend,sales,organic_sales,tacos
55,2026-03-16,1530.35,710.43,824.69,705.66,0.464227
56,2026-03-23,1294.42,268.70,532.80,761.62,0.207583
57,2026-03-30,1071.60,271.78,616.81,454.79,0.253621
58,2026-04-06,1405.58,394.35,685.81,719.77,0.280560
59,2026-04-13,1474.50,315.38,894.72,579.78,0.213889
60,2026-04-20,1400.50,295.73,649.80,750.70,0.211160
61,2026-04-27,1450.50,385.40,699.75,750.75,0.265701
62,2026-05-04,1571.47,408.48,639.79,931.68,0.259935
63,2026-05-11,2093.39,385.77,1417.64,675.75,0.184280
64,2026-05-18,1641.39,378.11,785.76,855.63,0.230360


In [45]:
ppc_df["brand_type"] = np.where(
    ppc_df["search_term"]
        .fillna("")
        .str.lower()
        .str.contains("litet"),
    "Branded",
    "Non-Branded"
)

In [46]:
brand_weekly = (
    ppc_df
    .groupby(
        [
            pd.Grouper(key="date", freq="W-MON"),
            "brand_type"
        ]
    )
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
)

brand_weekly["cvr"] = (
    brand_weekly["orders"] /
    brand_weekly["clicks"]
)

brand_weekly["roas"] = (
    brand_weekly["sales"] /
    brand_weekly["spend"]
)

brand_weekly

clicks   spend   sales  orders       cvr      roas
date       brand_type                                                     
2025-03-24 Non-Branded      15   38.32   13.99       1  0.066667  0.365084
2025-03-31 Non-Branded      54  112.31   55.96       4  0.074074  0.498264
2025-04-07 Branded           3    3.63    0.00       0  0.000000  0.000000
           Non-Branded      32   59.80   55.96       4  0.125000  0.935786
2025-04-14 Non-Branded      31   50.91  111.92       8  0.258065  2.198389
...                        ...     ...     ...     ...       ...       ...
2026-06-08 Branded           7   10.84   14.99       1  0.142857  1.382841
           Non-Branded     122  189.05  489.84      14  0.114754  2.591061
2026-06-15 Branded          12   16.48   94.97       3  0.250000  5.762743
           Non-Branded     250  571.83  709.81      19  0.076000  1.241295
2026-06-22 Non-Branded      43  105.30  134.96       4  0.093023  1.281671

[103 rows x 6 columns]

In [47]:
top_terms = (
    bad_period
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
    .sort_values("spend", ascending=False)
)

In [48]:
top_terms["cvr"] = top_terms["orders"] / top_terms["clicks"]
top_terms["acos"] = top_terms["spend"] / top_terms["sales"]

In [49]:
bad_period_non_brand = bad_period[
    ~bad_period["search_term"]
        .fillna("")
        .str.lower()
        .str.contains("litet")
]

In [50]:
waste_terms = (
    bad_period_non_brand
    .groupby("search_term")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
)

In [51]:
waste_terms["cvr"] = waste_terms["orders"] / waste_terms["clicks"]

In [52]:
waste_terms.sort_values(
    "spend",
    ascending=False
).head(50)

,clicks,spend,sales,orders,cvr
search_term,,,,,
cycling socks white,13,27.49,0.00,0,0.000000
cycling socks,17,25.92,0.00,0,0.000000
cycling socks for men,9,18.46,0.00,0,0.000000
white cycling socks,9,14.59,79.98,2,0.222222
white cycling socks men,5,9.72,0.00,0,0.000000
aero cycling socks,6,8.64,0.00,0,0.000000
aero socks cycling men,5,7.69,39.99,1,0.200000
mens cycling socks,4,7.63,0.00,0,0.000000
cycling socks men,3,6.18,0.00,0,0.000000


In [53]:
waste_terms[
    (waste_terms["spend"] > 5) &
    (waste_terms["orders"] == 0)
].sort_values(
    "spend",
    ascending=False
)

,clicks,spend,sales,orders,cvr
search_term,,,,,
cycling socks white,13,27.49,0.0,0,0.0
cycling socks,17,25.92,0.0,0,0.0
cycling socks for men,9,18.46,0.0,0,0.0
white cycling socks men,5,9.72,0.0,0,0.0
aero cycling socks,6,8.64,0.0,0,0.0
mens cycling socks,4,7.63,0.0,0,0.0
cycling socks men,3,6.18,0.0,0,0.0
long cycling socks,2,5.52,0.0,0,0.0


In [54]:
top_keywords = (
    ppc_df.groupby("keyword")
    .agg(spend=("spend","sum"))
    .sort_values("spend", ascending=False)
    .head(20)
    .index
)

In [55]:
good_keyword = (
    good_period[good_period["keyword"].isin(top_keywords)]
    .groupby("keyword")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
)

In [56]:
bad_keyword = (
    bad_period[bad_period["keyword"].isin(top_keywords)]
    .groupby("keyword")
    .agg(
        clicks=("clicks","sum"),
        spend=("spend","sum"),
        sales=("sales","sum"),
        orders=("orders","sum")
    )
)

In [57]:
for df in [good_keyword, bad_keyword]:
    df["cpc"] = df["spend"] / df["clicks"]
    df["cvr"] = df["orders"] / df["clicks"]
    df["roas"] = df["sales"] / df["spend"]
    df["acos"] = df["spend"] / df["sales"]

In [58]:
good_keyword = good_keyword.add_suffix("_good")
bad_keyword = bad_keyword.add_suffix("_bad")

In [59]:
keyword_compare = (
    good_keyword
    .merge(
        bad_keyword,
        left_index=True,
        right_index=True,
        how="outer"
    )
    .fillna(0)
)

In [60]:
keyword_compare["sales_change"] = (
    keyword_compare["sales_bad"]
    - keyword_compare["sales_good"]
)

keyword_compare["orders_change"] = (
    keyword_compare["orders_bad"]
    - keyword_compare["orders_good"]
)

keyword_compare["cvr_change"] = (
    keyword_compare["cvr_bad"]
    - keyword_compare["cvr_good"]
)

In [61]:
keyword_compare.sort_values(
    "sales_change"
).head(20)

,clicks_good,spend_good,sales_good,orders_good,cpc_good,cvr_good,roas_good,acos_good,clicks_bad,spend_bad,sales_bad,orders_bad,cpc_bad,cvr_bad,roas_bad,acos_bad,sales_change,orders_change,cvr_change
keyword,,,,,,,,,,,,,,,,,,,
cycling socks,116.0,201.01,629.83,17.0,1.732845,0.146552,3.133327,0.319150,60.0,83.14,79.98,2.0,1.385667,0.033333,0.961992,1.039510,-549.85,-15.0,-0.113218
white cycling socks,38.0,76.21,306.93,7.0,2.005526,0.184211,4.027424,0.248298,17.0,25.74,79.98,2.0,1.514118,0.117647,3.107226,0.321830,-226.95,-5.0,-0.066563
cycling socks white,7.0,16.84,200.95,5.0,2.405714,0.714286,11.932898,0.083802,15.0,34.27,0.00,0.0,2.284667,0.000000,0.000000,inf,-200.95,-5.0,-0.714286
road cycling socks men,12.0,27.96,119.97,3.0,2.330000,0.250000,4.290773,0.233058,15.0,25.49,0.00,0.0,1.699333,0.000000,0.000000,inf,-119.97,-3.0,-0.250000
"asin-expanded=""B0BGJNRR5D""",3.0,3.54,39.99,1.0,1.180000,0.333333,11.296610,0.088522,3.0,2.64,0.00,0.0,0.880000,0.000000,0.000000,inf,-39.99,-1.0,-0.333333
tall cycling socks,2.0,3.83,39.99,1.0,1.915000,0.500000,10.441253,0.095774,1.0,2.36,0.00,0.0,2.360000,0.000000,0.000000,inf,-39.99,-1.0,-0.500000
litet cycling sock,9.0,15.30,39.99,1.0,1.700000,0.111111,2.613725,0.382596,12.0,37.00,39.99,1.0,3.083333,0.083333,1.080811,0.925231,0.00,0.0,-0.027778
cycling socks men,2.0,2.14,0.00,0.0,1.070000,0.000000,0.000000,inf,0.0,0.00,0.00,0.0,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.000000
cycling socks for men,0.0,0.00,0.00,0.0,0.000000,0.000000,0.000000,0.000000,3.0,4.35,0.00,0.0,1.450000,0.000000,0.000000,inf,0.00,0.0,0.000000


In [62]:
keyword_compare[
    (keyword_compare["spend_good"] > 10) &
    (keyword_compare["spend_bad"] > 10)
][[
    "clicks_good",
    "clicks_bad",
    "sales_good",
    "sales_bad",
    "cpc_good",
    "cpc_bad",
    "cvr_good",
    "cvr_bad",
    "roas_good",
    "roas_bad"
]].sort_values(
    "sales_good",
    ascending=False
)

,clicks_good,clicks_bad,sales_good,sales_bad,cpc_good,cpc_bad,cvr_good,cvr_bad,roas_good,roas_bad
keyword,,,,,,,,,,
cycling socks,116.0,60.0,629.83,79.98,1.732845,1.385667,0.146552,0.033333,3.133327,0.961992
white cycling socks,38.0,17.0,306.93,79.98,2.005526,1.514118,0.184211,0.117647,4.027424,3.107226
cycling socks white,7.0,15.0,200.95,0.00,2.405714,2.284667,0.714286,0.000000,11.932898,0.000000
road cycling socks men,12.0,15.0,119.97,0.00,2.330000,1.699333,0.250000,0.000000,4.290773,0.000000
litet cycling sock,9.0,12.0,39.99,39.99,1.700000,3.083333,0.111111,0.083333,2.613725,1.080811
men cycling socks,6.0,27.0,39.99,54.98,1.775000,2.170741,0.166667,0.074074,3.754930,0.938065


In [63]:
import pandas as pd
from pathlib import Path

business_file = Path.cwd().parent / "BusinessReport-6-06-26.csv"

business = pd.read_csv(business_file)

print(business.shape)
business.head()

(14, 18)


,Date,Ordered Product Sales,Ordered Product Sales - B2B,Units Ordered,Units Ordered - B2B,Total Order Items,Total Order Items - B2B,Average Sales per Order Item,Average Sales per Order Item - B2B,Average Units per Order Item,Average Units per Order Item - B2B,Average Selling Price,Average Selling Price - B2B,Sessions - Total,Sessions - Total - B2B,Order Item Session Percentage,Order Item Session Percentage - B2B,Average Offer Count
0,3/1/26,$435.82,$0.00,18,0,17,0,$25.64,$0.00,1.06,0.0,$24.21,$0.00,272,8,6.25%,0.00%,37
1,3/8/26,"$1,626.33",$152.94,67,6,63,4,$25.81,$38.24,1.06,1.5,$24.27,$25.49,"1,085",49,5.81%,8.16%,37
2,3/15/26,"$1,364.37",$13.99,63,1,59,1,$23.12,$13.99,1.07,1.0,$21.66,$13.99,963,49,6.13%,2.04%,38
3,3/22/26,"$1,336.46",$0.00,57,0,50,0,$26.73,$0.00,1.14,0.0,$23.45,$0.00,911,23,5.49%,0.00%,38
4,3/29/26,"$1,457.50",$53.98,50,2,49,2,$29.74,$26.99,1.02,1.0,$29.15,$26.99,"1,033",30,4.74%,6.67%,38


In [64]:
business.columns = (
    business.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

business.columns.tolist()

['date',
 'ordered_product_sales',
 'ordered_product_sales___b2b',
 'units_ordered',
 'units_ordered___b2b',
 'total_order_items',
 'total_order_items___b2b',
 'average_sales_per_order_item',
 'average_sales_per_order_item___b2b',
 'average_units_per_order_item',
 'average_units_per_order_item___b2b',
 'average_selling_price',
 'average_selling_price___b2b',
 'sessions___total',
 'sessions___total___b2b',
 'order_item_session_percentage',
 'order_item_session_percentage___b2b',
 'average_offer_count']

In [65]:
business.head(3)

,date,ordered_product_sales,ordered_product_sales___b2b,units_ordered,units_ordered___b2b,total_order_items,total_order_items___b2b,average_sales_per_order_item,average_sales_per_order_item___b2b,average_units_per_order_item,average_units_per_order_item___b2b,average_selling_price,average_selling_price___b2b,sessions___total,sessions___total___b2b,order_item_session_percentage,order_item_session_percentage___b2b,average_offer_count
0,3/1/26,$435.82,$0.00,18,0,17,0,$25.64,$0.00,1.06,0.0,$24.21,$0.00,272,8,6.25%,0.00%,37
1,3/8/26,"$1,626.33",$152.94,67,6,63,4,$25.81,$38.24,1.06,1.5,$24.27,$25.49,"1,085",49,5.81%,8.16%,37
2,3/15/26,"$1,364.37",$13.99,63,1,59,1,$23.12,$13.99,1.07,1.0,$21.66,$13.99,963,49,6.13%,2.04%,38


In [66]:
business.tail(3)

,date,ordered_product_sales,ordered_product_sales___b2b,units_ordered,units_ordered___b2b,total_order_items,total_order_items___b2b,average_sales_per_order_item,average_sales_per_order_item___b2b,average_units_per_order_item,average_units_per_order_item___b2b,average_selling_price,average_selling_price___b2b,sessions___total,sessions___total___b2b,order_item_session_percentage,order_item_session_percentage___b2b,average_offer_count
11,5/17/26,"$1,895.32",$0.00,68,0,65,0,$29.16,$0.00,1.05,0.0,$27.87,$0.00,"1,040",17,6.25%,0.00%,40
12,5/24/26,"$1,196.62",$39.99,38,1,38,1,$31.49,$39.99,1.00,1.0,$31.49,$39.99,956,13,3.97%,7.69%,39
13,5/31/26,$814.69,$29.98,31,2,29,1,$28.09,$29.98,1.07,2.0,$26.28,$14.99,687,15,4.22%,6.67%,39


In [67]:
business["ordered_product_sales"] = (
    business["ordered_product_sales"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

business["sessions___total"] = (
    business["sessions___total"]
    .replace(",", "", regex=True)
    .astype(int)
)

business["order_item_session_percentage"] = (
    business["order_item_session_percentage"]
    .str.replace("%","")
    .astype(float)
)

In [68]:
business[
    [
        "date",
        "ordered_product_sales",
        "sessions___total",
        "order_item_session_percentage"
    ]
]

,date,ordered_product_sales,sessions___total,order_item_session_percentage
0,3/1/26,435.82,272,6.25
1,3/8/26,1626.33,1085,5.81
2,3/15/26,1364.37,963,6.13
3,3/22/26,1336.46,911,5.49
4,3/29/26,1457.50,1033,4.74
5,4/5/26,1534.51,995,4.82
6,4/12/26,1637.39,1005,5.67
7,4/19/26,1320.57,1022,4.11
8,4/26/26,1413.49,1019,4.51
9,5/3/26,1927.40,903,6.53


In [69]:
baseline_conversion = business.loc[
    business["date"].isin(["5/3/26","5/10/26","5/17/26"]),
    "order_item_session_percentage"
].mean()

recent_conversion = business.loc[
    business["date"].isin(["5/24/26","5/31/26"]),
    "order_item_session_percentage"
].mean()

print("Baseline:", baseline_conversion)
print("Recent:", recent_conversion)
print(
    "Change:",
    round((recent_conversion / baseline_conversion - 1) * 100, 1),
    "%"
)

Baseline: 6.06
Recent: 4.095
Change: -32.4 %


In [70]:
business["week"] = pd.to_datetime(business["date"])

business_weekly = business.rename(columns={
    "ordered_product_sales": "business_sales",
    "sessions___total": "sessions",
    "order_item_session_percentage": "conversion_rate"
})[
    [
        "week",
        "business_sales",
        "sessions",
        "conversion_rate"
    ]
]

/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_50746/1794986959.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  business["week"] = pd.to_datetime(business["date"])


In [71]:
weekly_full = weekly.merge(
    business_weekly,
    on="week",
    how="left"
)

In [72]:
weekly_summary = weekly_full[
    [
        "week",
        "revenue",
        "business_sales",
        "sessions",
        "conversion_rate",
        "spend",
        "sales",
        "organic_sales",
        "tacos",
        "cvr"
    ]
].copy()

In [73]:
def conversion_status(x):
    if x >= 5.5:
        return "GREEN"
    elif x >= 4.5:
        return "YELLOW"
    else:
        return "RED"


weekly_summary["conversion_status"] = weekly_summary["conversion_rate"].apply(conversion_status)

In [74]:
business["week"] = pd.to_datetime(business["date"]) + pd.Timedelta(days=1)

business_weekly = business.rename(columns={
    "ordered_product_sales": "business_sales",
    "sessions___total": "sessions",
    "order_item_session_percentage": "conversion_rate"
})[
    [
        "week",
        "business_sales",
        "sessions",
        "conversion_rate"
    ]
]

/var/folders/6v/r4t6qrjx0tz5cmlchf8bqs5h0000gn/T/ipykernel_50746/674621526.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  business["week"] = pd.to_datetime(business["date"]) + pd.Timedelta(days=1)


In [75]:
weekly_full = weekly.merge(
    business_weekly,
    on="week",
    how="left"
)

weekly_summary = weekly_full[
    [
        "week",
        "revenue",
        "business_sales",
        "sessions",
        "conversion_rate",
        "spend",
        "sales",
        "organic_sales",
        "tacos",
        "cvr"
    ]
].copy()

weekly_summary.tail(15)

,week,revenue,business_sales,sessions,conversion_rate,spend,sales,organic_sales,tacos,cvr
55,2026-03-16,1530.35,1364.37,963.0,6.13,710.43,824.69,705.66,0.464227,0.099291
56,2026-03-23,1294.42,1336.46,911.0,5.49,268.70,532.80,761.62,0.207583,0.116129
57,2026-03-30,1071.60,1457.50,1033.0,4.74,271.78,616.81,454.79,0.253621,0.126667
58,2026-04-06,1405.58,1534.51,995.0,4.82,394.35,685.81,719.77,0.280560,0.091787
59,2026-04-13,1474.50,1637.39,1005.0,5.67,315.38,894.72,579.78,0.213889,0.157895
60,2026-04-20,1400.50,1320.57,1022.0,4.11,295.73,649.80,750.70,0.211160,0.112426
61,2026-04-27,1450.50,1413.49,1019.0,4.51,385.40,699.75,750.75,0.265701,0.109005
62,2026-05-04,1571.47,1927.40,903.0,6.53,408.48,639.79,931.68,0.259935,0.095890
63,2026-05-11,2093.39,1916.34,1112.0,5.40,385.77,1417.64,675.75,0.184280,0.169014
64,2026-05-18,1641.39,1895.32,1040.0,6.25,378.11,785.76,855.63,0.230360,0.103604


In [76]:
def conversion_status(x):
    if pd.isna(x):
        return "NO DATA"
    elif x >= 5.5:
        return "GREEN"
    elif x >= 4.5:
        return "YELLOW"
    else:
        return "RED"

weekly_summary["conversion_status"] = weekly_summary["conversion_rate"].apply(conversion_status)

weekly_summary.tail(15)

,week,revenue,business_sales,sessions,conversion_rate,spend,sales,organic_sales,tacos,cvr,conversion_status
55,2026-03-16,1530.35,1364.37,963.0,6.13,710.43,824.69,705.66,0.464227,0.099291,GREEN
56,2026-03-23,1294.42,1336.46,911.0,5.49,268.70,532.80,761.62,0.207583,0.116129,YELLOW
57,2026-03-30,1071.60,1457.50,1033.0,4.74,271.78,616.81,454.79,0.253621,0.126667,YELLOW
58,2026-04-06,1405.58,1534.51,995.0,4.82,394.35,685.81,719.77,0.280560,0.091787,YELLOW
59,2026-04-13,1474.50,1637.39,1005.0,5.67,315.38,894.72,579.78,0.213889,0.157895,GREEN
60,2026-04-20,1400.50,1320.57,1022.0,4.11,295.73,649.80,750.70,0.211160,0.112426,RED
61,2026-04-27,1450.50,1413.49,1019.0,4.51,385.40,699.75,750.75,0.265701,0.109005,YELLOW
62,2026-05-04,1571.47,1927.40,903.0,6.53,408.48,639.79,931.68,0.259935,0.095890,GREEN
63,2026-05-11,2093.39,1916.34,1112.0,5.40,385.77,1417.64,675.75,0.184280,0.169014,YELLOW
64,2026-05-18,1641.39,1895.32,1040.0,6.25,378.11,785.76,855.63,0.230360,0.103604,GREEN


In [77]:
weekly_summary["sessions_wow"] = weekly_summary["sessions"].pct_change()
weekly_summary["conversion_wow"] = weekly_summary["conversion_rate"].pct_change()
weekly_summary["sales_wow"] = weekly_summary["business_sales"].pct_change()

weekly_summary.tail(10)

,week,revenue,business_sales,sessions,conversion_rate,spend,sales,organic_sales,tacos,cvr,conversion_status,sessions_wow,conversion_wow,sales_wow
60,2026-04-20,1400.50,1320.57,1022.0,4.11,295.73,649.80,750.70,0.211160,0.112426,RED,0.016915,-0.275132,-0.193491
61,2026-04-27,1450.50,1413.49,1019.0,4.51,385.40,699.75,750.75,0.265701,0.109005,YELLOW,-0.002935,0.097324,0.070364
62,2026-05-04,1571.47,1927.40,903.0,6.53,408.48,639.79,931.68,0.259935,0.095890,GREEN,-0.113837,0.447894,0.363575
63,2026-05-11,2093.39,1916.34,1112.0,5.40,385.77,1417.64,675.75,0.184280,0.169014,YELLOW,0.231451,-0.173047,-0.005738
64,2026-05-18,1641.39,1895.32,1040.0,6.25,378.11,785.76,855.63,0.230360,0.103604,GREEN,-0.064748,0.157407,-0.010969
65,2026-05-25,1690.39,1196.62,956.0,3.97,520.08,624.80,1065.59,0.307669,0.073801,RED,-0.080769,-0.364800,-0.368645
66,2026-06-01,1336.62,814.69,687.0,4.22,311.04,269.92,1066.70,0.232706,0.044944,RED,-0.281381,0.062972,-0.319174
67,2026-06-08,989.59,NaN,NaN,NaN,199.89,504.83,484.76,0.201993,0.116279,NO DATA,NaN,NaN,NaN
68,2026-06-15,1399.56,NaN,NaN,NaN,588.31,804.78,594.78,0.420354,0.083969,NO DATA,NaN,NaN,NaN
69,2026-06-22,459.82,NaN,NaN,NaN,105.30,134.96,324.86,0.229003,0.093023,NO DATA,NaN,NaN,NaN


In [78]:
hero = sales_df[
    sales_df["item_name"] == "3-pack White, Small/Medium"
]

In [79]:
hero_weekly = (
    hero
    .groupby(
        pd.Grouper(key="purchase_date", freq="W-MON")
    )
    .agg(
        revenue=("revenue","sum"),
        orders=("amazon-order-id","nunique"),
        pairs=("pairs_sold","sum")
    )
)

hero_weekly.tail(12)

,revenue,orders,pairs
purchase_date,,,
2026-04-06 00:00:00+00:00,599.85,15,45
2026-04-13 00:00:00+00:00,599.85,15,45
2026-04-20 00:00:00+00:00,519.87,11,39
2026-04-27 00:00:00+00:00,479.88,12,36
2026-05-04 00:00:00+00:00,679.83,16,51
2026-05-11 00:00:00+00:00,1119.72,28,84
2026-05-18 00:00:00+00:00,439.89,12,36
2026-05-25 00:00:00+00:00,639.84,16,48
2026-06-01 00:00:00+00:00,519.87,13,39


In [80]:
sales_df.groupby(
    [
        pd.Grouper(key="purchase_date", freq="W-MON"),
        "type"
    ]
).agg(
    revenue=("revenue","sum"),
    orders=("amazon-order-id","nunique")
).reset_index()

,purchase_date,type,revenue,orders
0,2025-02-24 00:00:00+00:00,single,69.95,5
1,2025-03-03 00:00:00+00:00,single,55.96,4
2,2025-03-10 00:00:00+00:00,single,265.81,19
3,2025-03-17 00:00:00+00:00,single,237.83,17
4,2025-03-24 00:00:00+00:00,single,41.97,3
...,...,...,...,...
130,2026-06-15 00:00:00+00:00,6-pack,79.99,1
131,2026-06-15 00:00:00+00:00,single,239.84,16
132,2026-06-22 00:00:00+00:00,3-pack,199.95,5
133,2026-06-22 00:00:00+00:00,6-pack,79.99,1


In [81]:
type_pivot = (
    sales_df.groupby(
        [
            pd.Grouper(key="purchase_date", freq="W-MON"),
            "type"
        ]
    )
    .agg(
        revenue=("revenue","sum")
    )
    .reset_index()
    .pivot(
        index="purchase_date",
        columns="type",
        values="revenue"
    )
)

type_pivot.tail(12)

type,3-pack,6-pack,single
purchase_date,,,
2026-04-06 00:00:00+00:00,1159.71,65.99,179.88
2026-04-13 00:00:00+00:00,1159.71,NaN,314.79
2026-04-20 00:00:00+00:00,959.76,65.99,374.75
2026-04-27 00:00:00+00:00,1039.74,65.99,344.77
2026-05-04 00:00:00+00:00,1079.73,131.98,359.76
2026-05-11 00:00:00+00:00,1559.61,263.96,269.82
2026-05-18 00:00:00+00:00,999.75,131.98,509.66
2026-05-25 00:00:00+00:00,1159.71,65.99,464.69
2026-06-01 00:00:00+00:00,959.76,211.97,164.89


In [82]:
sku_share = (
    sales_df[
        sales_df["purchase_date"] >= "2026-04-01"
    ]
    .groupby("item_name")
    .agg(
        revenue=("revenue","sum"),
        orders=("amazon-order-id","nunique")
    )
    .sort_values("revenue", ascending=False)
)

In [83]:
sku_share

,revenue,orders
item_name,,
"3-pack White, Small/Medium",6598.35,163
"3-pack White, Large/X-Large",3279.18,82
"White, Small/Medium",1079.28,69
"3-pack Black, Large/X-Large",1039.74,26
"3-pack Black, Small/Medium",959.76,23
"White, Large/X-Large",884.41,55
"6-pack White, Large/X-Large",833.88,12
"Blue, Small/Medium",524.65,26
"Blue, Large/X-Large",464.69,27


In [84]:
sales_df["period"] = np.where(
    sales_df["purchase_date"] < "2026-05-25",
    "Before",
    "After"
)

sku_mix = (
    sales_df
    .groupby(["period","item_name"])
    .agg(
        revenue=("revenue","sum")
    )
    .reset_index()
)

In [85]:
period_totals = sku_mix.groupby("period")["revenue"].transform("sum")

sku_mix["revenue_share"] = (
    sku_mix["revenue"] / period_totals
)

sku_mix.sort_values(
    ["item_name","period"]
)

,period,item_name,revenue,revenue_share
0,After,"3-pack Black, Large/X-Large",199.95,0.046119
12,Before,"3-pack Black, Large/X-Large",3406.27,0.077001
1,After,"3-pack Black, Small/Medium",199.95,0.046119
13,Before,"3-pack Black, Small/Medium",3222.11,0.072838
2,After,"3-pack White, Large/X-Large",1039.74,0.239818
14,Before,"3-pack White, Large/X-Large",7585.73,0.171480
3,After,"3-pack White, Small/Medium",1519.62,0.350503
15,Before,"3-pack White, Small/Medium",12754.01,0.288311
4,After,"6-pack White, Large/X-Large",305.96,0.070570
16,Before,"6-pack White, Large/X-Large",593.91,0.013426


In [86]:
after = sales_df[sales_df["purchase_date"] >= "2026-05-25"]
before = sales_df[
    (sales_df["purchase_date"] >= "2026-04-01") &
    (sales_df["purchase_date"] < "2026-05-25")
]

before_type = before.groupby("type").agg(
    revenue=("revenue", "sum"),
    orders=("amazon-order-id", "nunique")
)

after_type = after.groupby("type").agg(
    revenue=("revenue", "sum"),
    orders=("amazon-order-id", "nunique")
)

type_compare = before_type.merge(
    after_type,
    left_index=True,
    right_index=True,
    suffixes=("_before", "_after")
)

type_compare["revenue_share_before"] = type_compare["revenue_before"] / type_compare["revenue_before"].sum()
type_compare["revenue_share_after"] = type_compare["revenue_after"] / type_compare["revenue_after"].sum()

type_compare

,revenue_before,orders_before,revenue_after,orders_after,revenue_share_before,revenue_share_after
type,,,,,,
3-pack,8917.77,215,2959.26,72,0.714405,0.682559
6-pack,791.88,13,371.95,5,0.063438,0.085791
single,2773.15,153,1004.33,64,0.222158,0.231650
